# Notebook 04 — GDPR Article 9 Mapping

Maps biometric security research findings to UK GDPR Article 9 obligations for special category biometric data.

This notebook combines:
1. **Attack simulation** — presentation and digital attack vectors mapped to Art.32 security measures
2. **Full Art.9 assessment** — 22 checks across 7 obligation clusters with metric-linked scoring

**Art.9 framework:** Biometric data is always special category data. The satisfied threshold is fixed at **90%**.

**Metric-linked checks:** A9-022 (EER → DPIA risk), A9-042 (vulnerability → Art.32), A9-061 (EqOdds → discrimination).


## Part 1 — Attack vectors and Art.32 security mapping

Simulates 7 attack vectors (NIST FRVT PAD 2020; ISO/IEC 30107-3:2023) against three system configurations before the compliance assessment.


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from biometric_auth.engine.attack import run_attack_simulation
from biometric_auth.parsers.config_parser import load_config

CONFIG_DIR = Path('../data/sample_configs')
configs = {
    'Face (compliant)':  load_config(CONFIG_DIR / 'system_face_recognition.yaml'),
    'Fingerprint (partial)': load_config(CONFIG_DIR / 'system_fingerprint.yaml'),
    'Iris (research)':   load_config(CONFIG_DIR / 'system_iris.yaml'),
}
attack_results = {name: run_attack_simulation(cfg) for name, cfg in configs.items()}

for name, r in attack_results.items():
    print(f'{name}: overall={r.overall_vulnerability_score:.1f}/10, highest={r.highest_severity}')

In [ ]:
# Grouped bar chart: CVSS-like scores by attack vector, three systems
attack_ids = [v.attack_id for v in list(attack_results.values())[0].attack_vectors]
colours = ['#003087', '#F5A623', '#C8102E']
x = np.arange(len(attack_ids))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, r) in enumerate(attack_results.items()):
    scores = [v.cvss_like_score for v in r.attack_vectors]
    ax.bar(x + (i - 1) * w, scores, w, label=name, color=colours[i], alpha=0.85)

ax.axhline(7.0, color='red', linestyle='--', lw=1.2, label='HIGH threshold (7.0)')
ax.axhline(4.0, color='orange', linestyle='--', lw=1.2, label='MEDIUM threshold (4.0)')
ax.set_xticks(x); ax.set_xticklabels(attack_ids)
ax.set_ylabel('CVSS-like Score')
ax.set_title('Vulnerability Score by Attack Vector and System Configuration')
ax.legend(loc='upper right')
ax.set_ylim(0, 11)
plt.tight_layout(); plt.show()

In [ ]:
# Attack success rate table for the face system
face_r = attack_results['Face (compliant)']
rows = []
for v in face_r.attack_vectors:
    rows.append({'Attack': v.attack_id, 'Type': v.attack_type,
                 'Category': v.attack_category,
                 'Success Rate': f'{v.attack_success_rate:.1%}',
                 'CVSS-like': f'{v.cvss_like_score:.1f}',
                 'Severity': v.severity})
pd.DataFrame(rows)

## Part 2 — Full GDPR Art.9 compliance assessment

Runs the complete assessment on all three sample configurations and compares obligation-level scores.


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from biometric_auth.engine.attack import run_attack_simulation
from biometric_auth.engine.bias import run_bias_analysis
from biometric_auth.engine.gdpr import run_art9_assessment
from biometric_auth.engine.metrics import run_evaluation
from biometric_auth.parsers.config_parser import load_config
from biometric_auth.parsers.score_file import load_scores

CONFIG_DIR = Path('../data/sample_configs')
DATA_DIR   = Path('../data/synthetic')

# Use high-accuracy scores for face, medium for fingerprint, low for iris (representative)
system_data = {
    'Face (compliant)':       (load_config(CONFIG_DIR / 'system_face_recognition.yaml'),
                               load_scores(DATA_DIR / 'scores_high_accuracy.csv')),
    'Fingerprint (partial)':  (load_config(CONFIG_DIR / 'system_fingerprint.yaml'),
                               load_scores(DATA_DIR / 'scores_medium_accuracy.csv')),
    'Iris (research)':        (load_config(CONFIG_DIR / 'system_iris.yaml'),
                               load_scores(DATA_DIR / 'scores_low_accuracy.csv')),
}

reports = {}
for name, (cfg, df) in system_data.items():
    eval_r  = run_evaluation(df, algorithm_name=cfg.algorithm.name, modality=cfg.algorithm.modality)
    bias_r  = run_bias_analysis(df, algorithm_name=cfg.algorithm.name) if 'group' in df.columns else None
    atk_r   = run_attack_simulation(cfg, baseline_far=eval_r.far)
    reports[name] = run_art9_assessment(cfg, evaluation_result=eval_r,
                                         bias_result=bias_r, attack_result=atk_r)
    r = reports[name]
    print(f'{name}: score={r.overall_score:.1%} | status={r.overall_status} | risk={r.risk_rating}')

In [ ]:
# Obligation-level heatmap
obl_refs = [obl.obligation_ref for obl in list(reports.values())[0].obligations]
heatmap_data = np.array([
    [obl.score for obl in reports[name].obligations]
    for name in reports
])

fig, ax = plt.subplots(figsize=(11, 3.5))
im = ax.imshow(heatmap_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(obl_refs))); ax.set_xticklabels(obl_refs, fontsize=10)
ax.set_yticks(range(len(reports))); ax.set_yticklabels(list(reports.keys()))
plt.colorbar(im, ax=ax, label='Score (0=GAP, 1=SATISFIED)')
ax.set_title('GDPR Art.9 Obligation Scores by System\n(Green=SATISFIED, Red=GAP, threshold=90%)')

for i in range(len(reports)):
    for j in range(len(obl_refs)):
        ax.text(j, i, f'{heatmap_data[i,j]:.0%}', ha='center', va='center', fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# Summary table
summary = []
for name, r in reports.items():
    gaps = sum(1 for obl in r.obligations for c in obl.checks if c.status == 'GAP')
    crit = sum(1 for obl in r.obligations for c in obl.checks
               if c.status == 'GAP' and c.severity == 'CRITICAL')
    summary.append({
        'System': name, 'Score': f'{r.overall_score:.1%}',
        'Status': r.overall_status, 'Risk': r.risk_rating,
        'GAPs': gaps, 'CRITICAL GAPs': crit,
    })
pd.DataFrame(summary)

In [ ]:
# Most critical findings for the Iris (research) system
iris_r = reports['Iris (research)']
findings = [
    {'Check': c.check_id, 'Status': c.status, 'Severity': c.severity,
     'Finding': c.finding[:100]}
    for obl in iris_r.obligations
    for c in obl.checks
    if c.status == 'GAP'
]
findings.sort(key=lambda x: x['Check'])
pd.DataFrame(findings)